In [2]:
import openmeteo_requests
import pandas as pd
import numpy as np
from retry_requests import retry
import requests_cache

In [3]:
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

In [44]:
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
    "latitude": 47.1221,
	"longitude": 9.486,
	"start_date": "2022-01-01",
	"end_date": "2025-12-31",
	"daily": ["temperature_2m_max", "temperature_2m_min", "precipitation_sum", "temperature_2m_mean"],
	"timezone": "Europe/Berlin",
}

In [43]:
responses = openmeteo.weather_api(url=url, params=params)

In [45]:
response = responses[0]

In [46]:
print("Location: Sevelen SG, Switzerland")
print(f"Coordinates: {response.Latitude():.2f}°N {response.Longitude():.2f}°E")
print(f"Elevation: {response.Elevation()} m asl")

Location: Sevelen SG, Switzerland
Coordinates: 47.12°N 9.48°E
Elevation: 462.0 m asl


In [47]:
daily = response.Daily()

In [48]:
daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(2).ValuesAsNumpy()
daily_temperature_2m_mean = daily.Variables(3).ValuesAsNumpy()

In [49]:
daily_data = {
    "date": pd.date_range(
        start = pd.to_datetime(daily.Time(), unit = "s", utc=True),
        end = pd.to_datetime(daily.TimeEnd(), unit = "s", utc=True),
        freq = pd.Timedelta(seconds = daily.Interval()),
        inclusive="left"
    ).tz_convert(response.Timezone().decode())
}

In [50]:
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["temperature_2m_mean"] = daily_temperature_2m_mean

In [51]:
daily_dataframe = pd.DataFrame(data=daily_data)

In [52]:
daily_dataframe

,date,temperature_2m_max,temperature_2m_min,precipitation_sum,temperature_2m_mean
0,2021-12-31 23:00:00+01:00,10.212,-1.288,0.000000,3.591166
1,2022-01-01 23:00:00+01:00,11.762,0.012,0.000000,4.751583
2,2022-01-02 23:00:00+01:00,11.162,4.862,0.300000,8.414083
3,2022-01-03 23:00:00+01:00,11.412,5.162,0.100000,8.091167
4,2022-01-04 23:00:00+01:00,9.712,-5.538,15.900001,2.678666
...,...,...,...,...,...
1456,2025-12-26 23:00:00+01:00,-0.374,-3.374,0.000000,-2.336500
1457,2025-12-27 23:00:00+01:00,-1.924,-4.024,0.000000,-2.940666
1458,2025-12-28 23:00:00+01:00,-1.424,-4.624,0.000000,-3.286500
1459,2025-12-29 23:00:00+01:00,0.026,-3.324,0.000000,-1.494833
